# DCA Buy-the-Bear Final — Progressive Reinvestment (Optimum)

**Parameters:** Start 1%, +2% per red-month buy, cap 50%, reset after each 70% ATH sell.

Strategy: Buy $100 BTC + reinvest X% of USDT reserve every red monthly candle. Sell 70% at every new ATH.

In [ ]:
import pandas as pd
import requests
import time
import json
import numpy as np
from datetime import datetime, timezone

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: '%.6f' % x)

In [ ]:
def fetch_monthly_klines(symbol='BTCUSDT', limit=500):
    url = 'https://api.binance.com/api/v3/klines'
    all_klines = []
    start_time = 0
    while True:
        params = {'symbol': symbol, 'interval': '1M', 'startTime': start_time, 'limit': limit}
        resp = requests.get(url, params=params)
        data = resp.json()
        if not data or len(data) == 0:
            break
        all_klines.extend(data)
        print(f'Fetched {len(data)} months, up to {datetime.fromtimestamp(data[-1][0]/1000, tz=timezone.utc).strftime("%Y-%m")}')
        if len(data) < limit:
            break
        start_time = data[-1][0] + 1
        time.sleep(0.1)
    return all_klines

klines = fetch_monthly_klines()

In [ ]:
columns = ['timestamp', 'open', 'high', 'low', 'close', 'volume',
           'close_time', 'quote_vol', 'trades', 'taker_buy_base', 'taker_buy_quote', 'ignore']
df = pd.DataFrame(klines, columns=columns)
df['date'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
for c in ['open', 'high', 'low', 'close', 'volume']:
    df[c] = df[c].astype(float)
df = df[['date', 'open', 'high', 'low', 'close', 'volume']].copy()
df = df.sort_values('date').reset_index(drop=True)
print(f'{df["date"].min().strftime("%Y-%m")} → {df["date"].max().strftime("%Y-%m")} ({len(df)} months)')

In [ ]:
# Parameters (optimum from v5)
START_PCT = 1.0
INCREMENT = 2.0
CAP = 50.0

btc_held = 0.0
usdt_reserve = 0.0
total_invested = 0.0
highest_close = 0.0
reinvest_pct = START_PCT

records = []

for i, row in df.iterrows():
    close = row['close']
    month = row['date']
    action = 'nothing'

    # Entry: buy $10 + reinvest X% of reserve on red monthly candle
    if close < row['open']:
        extra = usdt_reserve * (reinvest_pct / 100.0) if usdt_reserve > 0 else 0.0
        total_usdt = 100.0 + extra
        btc_bought = total_usdt / close
        btc_held += btc_bought
        total_invested += 100.0
        usdt_reserve -= extra
        action = f'buy ${total_usdt:.1f} @ {close:.0f} (${10:.0f}+{reinvest_pct:.0f}%={extra:.1f})'

    prev_highest = highest_close
    if close > highest_close:
        highest_close = close

    # Exit: sell 70% at new ATH → reset reinvestment rate
    if btc_held > 0.000001 and close > prev_highest:
        sell_btc = btc_held * 0.7
        sell_usdt = sell_btc * close
        btc_held -= sell_btc
        usdt_reserve += sell_usdt
        reinvest_pct = START_PCT
        act = f'sell 70% @ {close:.0f} (-{sell_btc:.6f} BTC, +{sell_usdt:.1f} USDT)'
        action = f'{action} | {act}' if 'buy' in action else act
    else:
        if reinvest_pct < CAP:
            reinvest_pct = min(CAP, reinvest_pct + INCREMENT)

    portfolio_value = btc_held * close + usdt_reserve
    btc_value = btc_held * close

    records.append({
        'date': month,
        'close': close,
        'action': action,
        'reinvest_pct': reinvest_pct,
        'btc_held': btc_held,
        'btc_value': btc_value,
        'usdt_reserve': usdt_reserve,
        'total_invested': total_invested,
        'portfolio_value': portfolio_value,
    })

results = pd.DataFrame(records)
print(f'Total months: {len(results)}')
print(f'Trades with action: {len(results[results["action"] != "nothing"])}')

In [ ]:
# Metrics computation

final = results.iloc[-1]
last_close = final['close']
total_pnl = final['portfolio_value'] - final['total_invested']
ret_pct = (final['portfolio_value'] / final['total_invested'] - 1) * 100

# Max drawdown
eq = results['portfolio_value']
running_max = eq.cummax()
dd_raw = (running_max - eq) / running_max
max_dd = dd_raw.max() * 100

# Sharpe (annualized, skip 12-month warmup)
monthly_returns = eq.pct_change().dropna()
monthly_returns = monthly_returns[monthly_returns.index >= 12]
sharpe = (monthly_returns.mean() / monthly_returns.std()) * np.sqrt(12) if monthly_returns.std() > 0 else 0.0

# Profit factor
pos_sum = monthly_returns[monthly_returns > 0].sum()
neg_sum = monthly_returns[monthly_returns < 0].abs().sum()
pf = pos_sum / neg_sum if neg_sum > 0 else float('inf')

# Calmar
annualized_ret = (1 + ret_pct / 100) ** (12 / len(df)) - 1
calmar = annualized_ret / (max_dd / 100) if max_dd > 0 else 0

buys = [r for r in records if 'buy' in r['action']]
sells = [r for r in records if 'sell' in r['action']]

print('='*65)
print('PORTFOLIO SUMMARY — Progressive Reinvestment (Optimum)')
print('='*65)
print(f'Parameters:            start={START_PCT}%, +{INCREMENT}%/buy, cap={CAP}%')
print(f'Total invested:        {final["total_invested"]:.2f} USDT')
print(f'BTC held:              {final["btc_held"]:.8f} BTC')
print(f'BTC value:             {final["btc_value"]:.2f} USDT (at {last_close:.2f})')
print(f'USDT reserve:          {final["usdt_reserve"]:.2f} USDT')
print(f'Portfolio value:       {final["portfolio_value"]:.2f} USDT')
print(f'Total P&L:             {total_pnl:.2f} USDT')
print(f'Return:                {ret_pct:.2f}%')
print(f'Max drawdown:          {max_dd:.2f}%')
print(f'Sharpe ratio:          {sharpe:.2f}')
print(f'Profit factor:         {pf:.2f}')
print(f'Calmar ratio:          {calmar:.2f}')
print(f'Months:                {len(results)}')
print(f'Buy months:            {len(buys)}')
print(f'Sell months:           {len(sells)}')

In [ ]:
# Chart 1: Portfolio growth with buy/sell markers + drawdown

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker

fig = plt.figure(figsize=(14, 10))
gs = fig.add_gridspec(4, 1, height_ratios=[3, 1, 1.2, 1.2])

# --- Top panel: Portfolio value vs invested ---
ax1 = fig.add_subplot(gs[0])
ax1.fill_between(results['date'], results['total_invested'], results['portfolio_value'],
                 where=results['portfolio_value'] >= results['total_invested'],
                 color='green', alpha=0.12, label='Profit')
ax1.fill_between(results['date'], results['portfolio_value'], results['total_invested'],
                 where=results['portfolio_value'] < results['total_invested'],
                 color='red', alpha=0.12, label='Loss')
ax1.plot(results['date'], results['total_invested'], color='gray', linewidth=1.5, linestyle='--', label='Total Invested')
ax1.plot(results['date'], results['portfolio_value'], color='#2563eb', linewidth=2, label='Portfolio Value')

buys_df = results[results['action'].str.contains('buy', na=False)]
sells_df = results[results['action'].str.contains('sell', na=False)]
ax1.scatter(buys_df['date'], buys_df['portfolio_value'], color='#16a34a', s=25, marker='^', zorder=5, alpha=0.7, label='Buy')
ax1.scatter(sells_df['date'], sells_df['portfolio_value'], color='#dc2626', s=50, marker='v', zorder=5, label='Sell 70%')

ax1.set_title('DCA Buy-the-Bear — Progressive Reinvestment ($100/mo) (1% start, +2%/buy, cap 50%)', fontsize=13, fontweight='bold')
ax1.set_ylabel('USDT')
ax1.legend(loc='upper left', fontsize=8, ncol=3)
ax1.grid(True, alpha=0.25)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# --- 2nd panel: Drawdown ---
ax2 = fig.add_subplot(gs[1])
ax2.fill_between(results['date'], 0, dd_raw * 100, color='#dc2626', alpha=0.35)
ax2.plot(results['date'], dd_raw * 100, color='#dc2626', linewidth=0.8)
ax2.axhline(y=-max_dd, color='#991b1b', linestyle=':', linewidth=1.2, alpha=0.8, label=f'Max DD: {max_dd:.1f}%')
ax2.set_ylabel('Drawdown (%)')
ax2.legend(loc='lower left', fontsize=8)
ax2.grid(True, alpha=0.25)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# --- 3rd panel: BTC value vs USDT reserve stacked ---
ax3 = fig.add_subplot(gs[2])
ax3.fill_between(results['date'], 0, results['btc_value'], color='#f59e0b', alpha=0.6, label='BTC Value')
ax3.fill_between(results['date'], results['btc_value'], results['btc_value'] + results['usdt_reserve'],
                 color='#3b82f6', alpha=0.4, label='USDT Reserve')
ax3.plot(results['date'], results['portfolio_value'], color='#1e293b', linewidth=1, linestyle='--', label='Total')
ax3.set_ylabel('USDT')
ax3.legend(loc='upper left', fontsize=8, ncol=3)
ax3.grid(True, alpha=0.25)
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# --- 4th panel: Reinvestment rate over time ---
ax4 = fig.add_subplot(gs[3])
ax4.step(results['date'], results['reinvest_pct'], where='post', color='#8b5cf6', linewidth=1.5)
ax4.scatter(sells_df['date'], [START_PCT] * len(sells_df), color='#dc2626', s=40, marker='v', zorder=5, label='Reset on Sell')
ax4.set_ylabel('Reinvest Rate (%)')
ax4.set_xlabel('Date')
ax4.set_ylim(0, CAP * 1.15)
ax4.legend(loc='upper left', fontsize=8)
ax4.grid(True, alpha=0.25)
ax4.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig('dca-bt-buy-bot-100usd-progressive.png', dpi=150, bbox_inches='tight')
print('Chart saved as dca-bt-buy-bot-100usd-progressive.png')
plt.show()

In [ ]:
# Chart 2: BTC holdings over time

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ax1.fill_between(results['date'], 0, results['btc_held'], color='#f59e0b', alpha=0.5, step='post')
ax1.step(results['date'], results['btc_held'], where='post', color='#d97706', linewidth=1.5)
ax1.set_ylabel('BTC Holdings')
ax1.set_title('BTC Accumulation Over Time')
ax1.grid(True, alpha=0.25)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax2.step(results['date'], results['reinvest_pct'], where='post', color='#8b5cf6', linewidth=1.5)
ax2.axhline(y=CAP, color='#8b5cf6', linestyle=':', alpha=0.5, label=f'Cap: {CAP}%')
ax2.set_ylabel('Reinvest Rate (%)')
ax2.set_xlabel('Date')
ax2.set_ylim(0, CAP * 1.15)
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(True, alpha=0.25)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig('dca-bt-buy-bot-100usd-btc-holdings.png', dpi=150, bbox_inches='tight')
print('Chart saved as dca-bt-buy-bot-100usd-btc-holdings.png')
plt.show()

In [ ]:
# All trades table

pd.set_option('display.max_rows', None)
with_actions = results[results['action'] != 'nothing'].copy()
with_actions[['date', 'close', 'reinvest_pct', 'action', 'btc_held', 'btc_value', 'usdt_reserve', 'total_invested', 'portfolio_value']]